In [2]:
import os
from datetime import datetime
import sys
from pathlib import Path
import shutil
import zipfile
import pandas as pd
import tqdm
from tqdm.auto import tqdm
from thefuzz import process

In [3]:
def basic_file(rc1, df):
    rc1.loc[0, 'REJ'] = 0
    rc1.loc[0, 'RC'] = 1
    rc1.loc[0, 'ST'] = df.loc[0, 'Sample']
    rc1.loc[0, 'YR'] = yr
    rc1.loc[0, 'SESON'] = df.loc[0, 'Season Code']
    #rc1.loc[0, 'NOVP'] = df.loc[0, 'INPUT VALUE']
    #rc1.loc[0, 'NOVCOD'] = df.loc[0, 'INPUT VALUE']
    rc1.loc[0, 'STAT'] = df.loc[0, 'State Code']
    rc1.loc[0, 'DIST'] = df.loc[0, 'District Code']
    rc1.loc[0, 'STRA'] = df.loc[0, 'Stratum No']
    rc1.loc[0, 'VILL'] = df.loc[0, 'Order Of Selection']
    #rc1.loc[0, 'EPC'] = 
    rc1.loc[0, 'NOVPC'] = df.loc[0, 'Number Of Village Total']
    rc1.loc[0, 'NOVTRS'] = df.loc[0, 'Number Of Village TRS']

    return rc1

In [4]:
def comment_box_file(rc1, df):
    reason_map = {
                    'system of girdawari does not exist' : 0,
                    'girdawari not done for previous year/current year' : 1,
                    'khasra register/other records or statements (specify) not available' : 2,
                    'girdawari completed but jinswar/trs statement not prepared' : 3,
                    'reason for non availability of information not known' : 4,
                    'sample village is non trs village' : 5,
                    'not applicable for kerala, orissa and hilly districts of uttar pradesh' : 6,
                    'aggregation figures not available at village level for kerala and orissa' : 7,
                    'other reasons (specify)' : 8,
                    '' : 9
                }
    
    rc61_reason = str(df.iloc[0, 1]).lower()
    if not rc61_reason or rc61_reason.strip() == "":
        rc1.loc[0, 'RC61'] = reason_map['']
    else:
        best_match_key, score = process.extractOne(rc61_reason.strip(), reason_map.keys())
        if score >= 60:
            rc61_code = reason_map[best_match_key]
        else:
            rc61_code = 9
        rc1.loc[0, 'RC61'] = rc61_code
    
    rc62_reason = str(df.iloc[0, 1]).lower()
    if not rc62_reason or rc62_reason.strip() == "":
        rc1.loc[0, 'RC62'] = reason_map['']
    else:
        best_match_key, score = process.extractOne(rc62_reason.strip(), reason_map.keys())
        if score >= 60:
            rc62_code = reason_map[best_match_key]
        else:
            rc62_code = 9
        rc1.loc[0, 'RC62'] = rc62_code
    
    rc63_reason = str(df.iloc[0, 1]).lower()
    if not rc63_reason or rc63_reason.strip() == "":
        rc1.loc[0, 'RC63'] = reason_map['']
    else:
        best_match_key, score = process.extractOne(rc63_reason.strip(), reason_map.keys())
        if score >= 60:
            rc63_code = reason_map[best_match_key]
        else:
            rc63_code = 9
        rc1.loc[0, 'RC63'] = rc63_code
    
    return rc1

In [ ]:
def basic_file_block_11(rc1, df):
    rc1.loc[0, 'MAC'] = df.loc[0, 'mawp']
    if df.loc[0, 'mawp'] == 1:
        rc1.loc[0, 'MUC'] = df.loc[0, 'usable']
    
    cadc = df.loc[0, 'cs']
    rc1.loc[0, 'CADC'] = cadc
    if cadc in (3,4,9):
        rc1.loc[0, 'TLC'] = 0
    elif cadc in (1,2):
        map_upd_year = df.loc[0, 'muy']
        if map_upd_year == None:
            rc1.loc[0, 'TLC'] = 9
        else:
            # calculating number of years from map update date
            if len(str(map_upd_year))>4:
                map_upd_year = str(map_upd_year)[:4] + 1
            
            yr_gap = yr - map_upd_year
            if yr_gap <= 1:
                rc1.loc[0, 'TLC'] = 1
            elif yr_gap <= 5:
                rc1.loc[0, 'TLC'] = 2
            elif yr_gap <= 10:
                rc1.loc[0, 'TLC'] = 3
            elif yr_gap <= 20:
                rc1.loc[0, 'TLC'] = 4
            elif yr_gap > 20:
                rc1.loc[0, 'TLC'] = 5
    
    #############################################
    #### SGC, DDG, ADG, GCC
    #############################################
    ddg_str = str(df.loc[0, 'gdoc'])
    sgc = df.loc[0, 'gsoc']
    adg_str = str(df.loc[0, 'actualdate'])
    
    ddg_dt = None
    adg_dt = None
    if ddg_str != '':
        ddg_dt = datetime.strptime(ddg_str, "%d/%m/%Y")
    if adg_str != '':
        adg_dt = datetime.strptime(adg_Str, "%d/%m/%Y")
    
    rcfgnc = df['grfnc']    #reason code for girdawari not completed 10(d)
    
    # calculation of GCC
    if sgc == 1:
        if adg_str == '':
            gcc = 3
        elif adg_dt <= ddg_dt:
            gcc = 1
        elif adg_dt > ddg_dt:
            gcc = 2
    elif sgc in (2,3):
        if adg_str == '':
            if rcfgnc == 1:
                gcc = 4
            elif rcfgnc == 2:
                gcc = 5
            elif rcfgnc == 9:
                gcc = 6
            elif rcfgnc == None:
                gcc = 7
    elif sgc == None and adg_str == '':
        gcc = 9
    else:
        gcc = 8
    
    if sgc == 1:
        rc1.loc[0, 'EPC'] = 1
    else:
        rc1.loc[0, 'EPC'] = 3
    rc1.loc[0, 'SGC'] = sgc
    rc1.loc[0, 'DDG'] = f"{ddg_str[:2]}{ddg_str[3:5]}"
    rc1.loc[0, 'GCC'] = gcc
    rc1.loc[0, 'LFOG'] = df.loc[0, 'lfogu']
    rc1.loc[0, 'ROG'] = df.loc[0, 'rogk']
    
    #############################################
    #### DDTRS, ADTRS, TRSSF, TRSSC
    #############################################
    ddtrs_str = str(df.loc[0, 'ddfst'])
    trs = df.loc[0, 'tssta']
    adtrs_str = str(df.loc[0, 'iyados'])
    
    ddtrs_dt = None
    adtrs_dt = None
    if ddtrs_str != '':
        ddtrs_dt = datetime.strptime(ddtrs_str, "%d/%m/%Y")
    if adtrs_str != '':
        adtrs_dt = datetime.strptime(adtrs_str, "%d/%m/%Y")
    
    if ddtrs_str != '' and adtrs_str != '':
        if adtrs_dt <= ddtrs_dt:
            if adtrs_dt < adg_dt or (adg_str == '' and sgc in (2,3) ):
                trssc = 0
            elif adtrs_dt >= adg_dt:
                trssc = 1
            elif sgc == 1 and adg_str == '':
                trssc = 2
        elif adtrs_dt > ddtrs_dt:
            if (adg_str == '' or adtrs_dt < adg_dt) and sgc in (2,3):
                trssc = 3
            elif adtrs_dt >= adg_dt:
                trssc = 4
            elif sgc == 1 and adg_str == '':
                trssc = 5
    elif adtrs_str == '':
        if trs == 1:
            trssc = 6
    elif trs == 0:
        if sgc == 1:
            trssc = 7
        elif sgc in (2,3):
            trssc = 8
    elif (trs in ('', None) and adtrs_str == ''):
        trssc = 9
    elif sgc == 1 and trs == 0 and adtrs_dt < ddtrs_dt:
        trssc = 10
    
    rc1.loc[0, 'TRSSC'] = trssc
    rc1.loc[0, 'DDTRS'] = f"{ddtrs_str[:2]}{ddtrs_str[3:5]}"
    rc1.loc[0, 'TRSSF'] = df.loc[0, 'wteswsisf']
    rc1.loc[0, 'DCKC'] = df.loc[0, 'dcot']
    
    return rc1

In [ ]:
def basic_file_block_32(rc1, df):
    geoa_hec = df.loc[0, 'ih1']
    geoa_lu = df.loc[0, 'ilu11']
    if geoa_hec is not None:
        rc1.loc[0, 'GEOA'] = geoa_hec
    else:
        rc1.loc[0, 'GEOA'] = geoa_lu
    return rc1

In [ ]:
def get_rc1(csv_file, rc1, df):
    if csv_file.name == "BasicFile.csv":
        rc1 = basic_file(rc1, df)
    if csv_file.name == "commentboxfile6.csv":
        rc1 = comment_box_file(rc1, df)
    if csv_file.name == "basicfileblock11.csv":
        rc1 = basic_file_block_11(rc1, df)
    if csv_file.name == "basicfileblock32.csv":
        basic_file_block_32(rc1, df)
    
    if csv_file.name == "basicfileblock2.csv":
        rc1.loc[0, 'HSSN'] = df.loc[0, 'HSN']
    if csv_file.name == "basicfileblock33.csv":
        rc1.loc[0, 'TNSSN'] = df.loc[:, 'confirmation'].value_counts()['yes']
    return rc1

In [ ]:
def get_rc2(csv_file, rc2, df):
    return rc2

In [ ]:
def get_rc3(csv_file, rc3, df):
    return rc3

In [ ]:
def get_rc123(zip_files_path, curr_dir):
    csv_dir = curr_dir / "csv_files"
    os.makedirs(csv_dir, exist_ok=True)
    with zipfile.ZipFile(zip_file, 'r') as zf:
        zf.extractall(csv_dir)
    
    uid = ''
    yr = 0
    for csv_file in csv_dir.iterdir():
        df = pd.read_csv(csv_file)
        
        if csv_file.name == "BasicFile.csv":
            year = str(df.loc[0, 'Year'])
            yr = int(year[:4]) + 1
            state = str(df.loc[0, 'State Code'])
            season = str(df.loc[0, 'Season Code'])
            st = str(df.loc[0, 'Sample'])
            distt = str(df.loc[0, 'District Code'])
            stra = str(df.loc[0, 'Stratum No'])
            vill = str(df.loc[0, 'Order Of Selection'])
    
            unique_id = f"{yr}_{state}_{season}_{st}_{distt}_{stra}_{vill}"
            rc1.loc[0, 'ID'] = unique_id
            rc2.loc[0, 'ID'] = unique_id
            rc3.loc[0, 'ID'] = unique_id
        
        rc1 = get_rc1(csv_file, rc1, df)
        rc2 = get_rc2(csv_file, rc2, df)
        rc3 = get_rc3(csv_file, rc3, df)
            
    shutil.rmtree(csv_dir)
    return rc1, rc2, rc3

In [ ]:
rc1 = pd.DataFrame(columns=["RC","ST","YR","SESON","NOVP","NOVCOD","STAT"
            ,"DIST","STRA","VILL","EPC","NOVPC","NOVTRS","CADC"
            ,"TLC","MAC","MUC","DDG","SGC","GCC","LFOG","TCROPA"
            ,"ROG","DDTRS","TRSSC","TRSSF","DCKC","HSSN","GEOA"
            ,"TNSSN","TGEOASN","RC61","RC62","RC63","REJ","ID"])

rc2 = pd.DataFrame(columns=["RC","ST","YR","SESON","STAT","DIST","STRA"
                    ,"VILL","EPC","HSSN","TNSSN","CROP","VARC","ARSU"
                    ,"ARSI","ARPU","ARPI","ID"])

rc3 = pd.DataFrame(columns=["RC","ST","YR","SESON","STAT","DIST","STRA"
                    ,"VILL","SN","CROP","VARC","IRRC","ERC","ID"])

rc11 = rc1.copy()
rc22 = rc2.copy()
rc33 = rc3.copy()

curr_path = Path.cwd()
for zip_file in zip_files_path.iterdir():
    rc11, rc22, rc33 = get_rc123(zip_files_path, curr_path, rc1, rc2, rc3)
    

In [ ]:
def create_rc123(basic_zip_dir_path, rc1, rc2, rc3):
    # list of all files
    zip_files = os.listdir(basic_zip_dir_path)
    
    # for each zip file, create unique ID, create RC1, create RC2, create RC3
    processed_zip_path = basic_zip_dir_path / "processed_zip"
    os.makedirs(move_zip_to, exist_ok=True)
    for zip_file in zip_files:
        csv_files_dir = basic_zip_dir_path / "csv_files"
        os.makedirs(new_folder, exist_ok=True)
        
        # extract zip file basic_zip_dir_path
        with zipfile.ZipFile(zip_file, 'r') as zf:
            zf.extractall(csv_files_dir)
        # move zip to processed path
        shutil.move(zip_file, processed_zip_path)

        csv_files = os.listdir(csv_files_dir)
        for csv_file in csv_files:
            if csv_file.name == 'BasicFile.csv'


        shutil.rmtree(new_folder)
    

    # create unique_id

    # create rc1

    # create rc2

    # create rc3
    
    return rc1, rc2, rc3

In [ ]:
def main():
    dir_path = Path.cwd()
    parent_dir = Path.cwd() / "01_parent_directory"
    unnecessary_files_path = Path.cwd() / "02_unnecessary_files"
    basic_zip_dir_path = Path.cwd() / "03_basic_zip_files"
    aggregate_basic_zip(dir_path, basic_zip_dir_path, irrelevant_path)
    
    rc1 = pd.DataFrame(columns=["RC","ST","YR","SESON","NOVP","NOVCOD","STAT"
                        ,"DIST","STRA","VILL","EPC","NOVPC","NOVTRS","CADC"
                        ,"TLC","MAC","MUC","DDG","SGC","GCC","LFOG","TCROPA"
                        ,"ROG","DDTRS","TRSSC","TRSSF","DCKC","HSSN","GEOA"
                        ,"TNSSN","TGEOASN","RC61","RC62","RC63","REJ","ID"])
    rc2 = pd.DataFrame(columns=["RC","ST","YR","SESON","STAT","DIST","STRA"
                        ,"VILL","EPC","HSSN","TNSSN","CROP","VARC","ARSU"
                        ,"ARSI","ARPU","ARPI","ID"])
    rc3 = pd.DataFrame(columns=["RC","ST","YR","SESON","STAT","DIST","STRA"
                        ,"VILL","SN","CROP","VARC","IRRC","ERC","ID"])

    create_rc123(basic_zip_dir_path, rc1, rc2, rc3)